# PQD CNN Classifier. Training, Saving, Loading, Evaluating and TFLite Converting

In [1]:
import sys
print(sys.executable)

C:\Users\nxf93627\Projects\PQD-Classifier\.venv\Scripts\python.exe


In [2]:
import tensorflow as tf
import numpy as np
import os
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score

import config
import dataset
import models
import training
import utils

In [4]:
def get_file_size(file_path):
    size = os.path.getsize(file_path)
    return size
    
def convert_bytes(size, unit=None):
    if unit == "KB":
        return print('File size: ' + str(round(size / 1024, 3)) + ' Kilobytes')
    elif unit == "MB":
        return print('File size: ' + str(round(size / (1024 * 1024), 3)) + ' Megabytes')
    else:
        return print('File size: ' + str(size) + ' bytes')

def representative_dataset():    
    for i in range(512):
        yield [X_train[i:i+1].astype(np.float32)]

## Loading Model

In [5]:
settings = config.load_config('base_config.yaml')

X_train, Y_train, X_test, Y_test, encoder, metadata = dataset.loadMatlabDataset(settings['dataset'])

model_cfg = settings['model']
model_cfg.update({'input_shape': (metadata['observations'], 1), 'outputs': metadata['classes']})

model = models.build_model(settings['model'])

experiment_path = utils.set_timestamp_dir()
weights_path = utils.MODELS_DIR / 'run_08_15-13_56' / 'model.weights.h5'

Reading config yaml
Loading dataset from C:\Users\nxf93627\PQD Classifier\data\16pqd_480pattern_50hz_10cycle_noNoise.mat
Successfully read 7680 samples
Encoding labels with OneHot
Successfully encoded 16 categories
Splitted training (0.9) and testing sets (0.1)


In [6]:
model.load_weights(weights_path)
model = training.evaluate(model, X_test, Y_test, metadata['categories'])

C:\Users\nxf93627\Projects\PQD-Classifier\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'nadam', because it has 2 variables whereas the saved optimizer has 47 variables. 
  saveable.load_own_variables(store)


24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step
compile_metrics: 80.46875
accuracy: 0.7955729166666666
macro_f1: 0.7979322217493006
weighted_f1: 0.7979322217493007
classification_report:                         precision    recall  f1-score   support

               Flicker       0.94      0.65      0.77        48
     Flicker+Harmonics       1.00      1.00      1.00        48
           Flicker+Sag       0.96      0.94      0.95        48
         Flicker+Swell       0.73      0.98      0.84        48
             Harmonics       0.92      1.00      0.96        48
   Impulsive Transient       1.00      0.69      0.81        48
          Interruption       0.52      0.50      0.51        48
Interruption+Harmonics       0.44      0.48      0.46        48
                Normal       0.83      1.00      0.91        48
                 Notch       0.98      0.98      0.98        48
 Oscillatory transient       1.00      0.92      0.96        48
                   Sag       0.49      0.35      0.4

C:\Users\nxf93627\Projects\PQD-Classifier\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## TFLite Conversion

In [7]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

converter.representative_dataset = representative_dataset

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

quantized_model = converter.convert()

model_path = experiment_path / "model.tflite"
with open(model_path, "wb") as f:
    f.write(quantized_model)

convert_bytes(get_file_size(model_path), "MB")

INFO:tensorflow:Assets written to: C:\Users\nxf93627\AppData\Local\Temp\tmpbmkbfl5a\assets


INFO:tensorflow:Assets written to: C:\Users\nxf93627\AppData\Local\Temp\tmpbmkbfl5a\assets


Saved artifact at 'C:\Users\nxf93627\AppData\Local\Temp\tmpbmkbfl5a'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 640, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 16), dtype=tf.float32, name=None)
Captures:
  1739431266832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1738820041808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1738820041040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1738820039888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1738820042000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1738820042192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1738820042576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1738820040080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1738820040464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1738820040656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  173882

C:\Users\nxf93627\Projects\PQD-Classifier\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


File size: 0.191 Megabytes


In [8]:
# https://medium.com/@adityamohiteakm/understanding-post-training-quantization-ptq-for-edge-ai-deployment-a-beginners-guide-with-6d6ba596e8f5

X_testi = (X_test - X_test.min()) / (X_test.max() - X_test.min())
X_testi *= 255.0
X_testi = np.array(X_testi, dtype=np.uint8)

interpreter = tf.lite.Interpreter(model_path = experiment_path / "model.tflite")
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

interpreter.resize_tensor_input(input_details[0]['index'], (len(Y_test), metadata['observations'], 1))
interpreter.resize_tensor_input(output_details[0]['index'], (len(Y_test), 16))
interpreter.allocate_tensors()

interpreter.set_tensor(input_details[0]['index'], X_testi)
interpreter.invoke()

predictions = interpreter.get_tensor(output_details[0]['index'])
prediction_classes = np.argmax(predictions, axis=1)

C:\Users\nxf93627\Projects\PQD-Classifier\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [9]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score

In [10]:
accuracy = accuracy_score(Y_test.argmax(axis=1), prediction_classes)
macro_f1 = f1_score(Y_test.argmax(axis=1), prediction_classes, average="macro")
weighted_f1 = f1_score(Y_test.argmax(axis=1), prediction_classes, average="weighted")

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro F1-score: {macro_f1:.4f}")
print(f"Weighted F1-score: {weighted_f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    Y_test.argmax(axis=1),
    prediction_classes,
    target_names=metadata['categories']
))

Accuracy: 0.8424
Macro F1-score: 0.8401
Weighted F1-score: 0.8401

Classification Report:
                        precision    recall  f1-score   support

               Flicker       0.96      1.00      0.98        48
     Flicker+Harmonics       1.00      1.00      1.00        48
           Flicker+Sag       0.98      1.00      0.99        48
         Flicker+Swell       1.00      0.94      0.97        48
             Harmonics       0.92      1.00      0.96        48
   Impulsive Transient       0.98      0.90      0.93        48
          Interruption       0.43      0.44      0.43        48
Interruption+Harmonics       0.47      0.33      0.39        48
                Normal       0.84      1.00      0.91        48
                 Notch       0.98      1.00      0.99        48
 Oscillatory transient       1.00      0.98      0.99        48
                   Sag       0.41      0.40      0.40        48
         Sag+Harmonics       0.49      0.60      0.54        48
             